In [0]:
 The 5 v of Big data

 velocity - speed of the data
 volume- huge data(real time vs batch)
 value - turning raw data into big
 veracity - data quality and reliablity
  variety - manu types

Real- world

social media
e-commerce
Iot- sensor data from devices
Banking

Traditonal dat processing
RDBMS and single- machine tools
limited by hardwares
can't handle TB+ data easily

Modern way Data
sitributed computing:split work across machines
horizonatl scaling:add more nodes to handle grgowth
Hadoop,apache spark


How spark works

Driver:The controller,send tasks and track jobs
executors: Workers that run task sand store data
cluster manager: allocates resorces
(databricks,YARN etc)

JOBS -> stages -> TASK -> sparks splits

python = python + spark

In [0]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName('Spark DataFrames').getOrCreate()
print("Sparksession Created:",spark.version)

In [0]:
#Create Spark

data=[(1,"James","Smith","USA","CA"),
      (2,"Michael","Rose","USA","NY"),
      (3,"Robert","Williams","USA","CA"),
      (4,"Maria","Jones","USA","FL")]
column =["id","firstname","lastname","country","state"]
spark_df = spark.createDataFrame(data,schema=column)
display(spark_df)


In [0]:
print("counts in thr df",spark_df.count())

Real-Life USE CASE

--ingestion data from files
--Transforamtion ata scale
--writing to Delta tables

In [0]:
from pyspark.sql.types import StructField,StringType,IntegerType,StructType
schema = StructType([StructField("id",IntegerType(),True),
                     StructField("firstname",StringType(),True),
                     StructField("lastname",StringType(),True),
                     StructField("country",StringType(),True),
                     StructField("state",StringType(),True)])
data=[(1,"James","Smith","USA","CA"),
      (2,"Michael","Rose","USA","NY"),
      (3,"Robert","Williams","USA","CA"),
      (4,"Maria","Jones","USA","FL")]
spark_df = spark.createDataFrame(data,schema=schema)
display(spark_df)


Power toll for big data
Build scalable pipelines
learn dataframe
grasp crisp


what is mapreduce(old way)

hadoop vs spark
break the task into map and reduce(map)
disk- heavy(map)

why spark is better
in-memory computation(fast)
chain operation without writing
much easier API(especially with pyspark)


spark is a engine its not a store,data not a programming languagae
process data in a disribute the data 

cook(executor)
manager(driver program)
Kitchen(cluster/compute)
prderflow(DAG schedular)
Each Order(Task)



Apache spark(open source)


In [0]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName('Spark DataFrames').getOrCreate()
print("Sparksession Created:",spark.version)


In [0]:
data = {
        (1, "James", "Smith", "USA", "CA"),
        (2, "Michael", "Rose", "USA", "NY"),
        (3, "Robert", "Williams", "USA", "CA"),
        (4, "Maria", "Jones", "USA", "FL")
}
column = ["id", "firstname", "lastname", "country", "state"]
spark_df = spark.createDataFrame(data,column)

In [0]:
spark_df.select("firstname", "lastname").show()

In [0]:
c

In [0]:
display(spark_df.filter(spark_df.firstname == "James"))

In [0]:
data = [
    ("Bob", "HR", 40000, 4000),
    ("Anurag", "Data Engineer", 100000, 10000),
    ("Ankur", "IT", 510000, 51000),
    ("Alice", "IT", 50000, 5000),
    ("Ayushi", "HR", 30000, 3000),
    ("Shruti", "Finance", 650000, 65000),
    ("Stuti", "Recruiter", 150000, 15000),
    ("Shalini", "Finance", 850000, 85000)
]

columns = ["Name", "Department", "Salary", "Bonus"]

df = spark.createDataFrame(data, columns)

display(df)

In [0]:
from pyspark.sql.functions import col

df = df.withColumn("Bonus", (col("Salary") * 0.1).cast("int"))
display(df)

In [0]:
df.groupBy("Department").agg({"Salary": "avg"}).show()

In [0]:
# Dept mapping dataframe
dept_data = [("IT", 101), ("HR", 102), ("Finance", 103)]
dept_columns = ["Department", "Dept_Code"]

df_dept = spark.createDataFrame(dept_data, dept_columns)

# Join
joined_df = df.join(df_dept, on="Department", how="inner")

display(joined_df)

Why partotioning Matters

1.Data is split into partiitions
2.Too few - underuntilized CPU
3.Too many -overhead
4.Control with repartition
coalesesce()
Cache and persis
1. Avoid re-computing same Data frame
2, .cache()-> keeps in memory
3. persist-> choose storage level(eg, disk and memeory)
4. what is Skewed Data
one key has much more data than others
cause data pile-up on 1 
executoe->slow job
how to handle skew
1.salting keys(add random value)
2.increase partition
3.broadcast smaller table during joins

Delta table
1. Transcational storage format like acid for big data
2,supports schema enforcments + version
3. great data lakes


In [0]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("DeltaVersioningDemo") \
    .getOrCreate()

In [0]:
data = [
    ("Alice", "IT", 50000),
    ("Bob", "HR", 45000),
    ("Charlie", "IT", 60000)
]

cols = ["Name", "Department", "Salary"]

df = spark.createDataFrame(data, cols)
display(df)

In [0]:
df.write.format("delta").mode("overwrite").saveAsTable("employees_delta")

In [0]:
%sql
select * from employees_delta

In [0]:
%sql
describe history employees_delta

In [0]:
from pyspark.sql import Row

update_data = [Row(Name="Alice", Department="IT", salary=52000)]
df_update = spark.createDataFrame(update_data)
df_update.createOrReplaceTempView("updates_view")

spark.sql("""
MERGE INTO employees_delta AS target
USING updates_view AS source
ON target.Name = source.Name
WHEN MATCHED THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *
""")

In [0]:
%sql
describe history employees_delta

schema Evolution

In [0]:
# New data with extra column
new_data = [
    ("David", "Finance", 70000, 5)
]

cols_new = ["Name", "Department", "Salary", "Experience"]

df_new = spark.createDataFrame(new_data, cols_new)
display(df_new)

Conecpr iof merge schema

In [0]:
df_new.write.format("delta") \
  .option("mergeSchema", "true") \
  .mode("append") \
  .saveAsTable("employees_delta")